# Basic SQL ETL And Schema Walkthrough

This notebook shows the smallest useful pattern for pulling data from any SQL Server database, transforming it in Python, and writing the result back to a table in a target schema. It also explains the PricingLab schema shape and points at the current SQL Server DDL that can be imported into an ERD generator.

The important idea is that model code should not know the authentication mechanism. It asks for a SQLAlchemy engine, runs SQL, and writes rows. The shared engine helper can use local SQL auth, work SQL auth, or Entra token auth behind the same call.

## DDL And ERD Files

Use `tutorials/schema/pricing_useful_tables_ddl.sql` when an ERD tool wants plain SQL Server `CREATE TABLE` syntax. This file is a tested copy of `docs/pricing_useful_tables_ddl.sql`, so it stays aligned with the repository reference.

Use `docs/pricing_useful_tables_full_ddl.sql` when you want the fuller SQL Server contract with schemas, constraints, filtered indexes, and runtime views. That file also includes `pricing.V_CURRENT_RATE_PACKAGE`, `pricing_runtime.V_COMPILED_RATE_CELL`, `pricing_runtime.V_COMPILED_RATE_CELL_LEVEL`, and `pricing_runtime.V_COMPILED_1D_RATE_BAND`.

`pricing.PREDICT_CURRENT_RATE` lives in `db/migrations/V014__current_rate_prediction_proc.sql`; it is a stored procedure rather than a table, so most ERD tools will not draw it.

## Logical Data Flow

```mermaid
flowchart LR
    source[(Source SQL Server)] --> raw[raw.FREMTPL_RAW or model source table]
    raw --> manifest[mlops.DATASET_MANIFEST]
    manifest --> columns[mlops.DATASET_COLUMN]
    manifest --> splitset[mlops.CV_SPLIT_SET]
    splitset --> folds[mlops.CV_FOLD]
    splitset --> splitrows[mlops.CV_SPLIT_ROW]
    manifest --> run[mlops.MODEL_RUN]
    run --> rundataset[mlops.MODEL_RUN_DATASET]
    run --> runsplit[mlops.MODEL_RUN_SPLIT_SET]
    run --> metrics[mlops.MODEL_RUN_METRIC]
    run --> package[pricing.RATE_PACKAGE]
    model[pricing.MODEL] --> run
    model --> package
    package --> term[pricing.TERM]
    term --> cell[pricing.RATE_CELL]
    feature[pricing.FEATURE] --> levelset[pricing.FEATURE_LEVEL_SET]
    levelset --> level[pricing.FEATURE_LEVEL]
    cell --> celllevel[pricing.RATE_CELL_LEVEL]
    package --> deployment[pricing.MODEL_DEPLOYMENT]
    deployment --> runtime[pricing_runtime compiled views]
```


## Schema Meaning

| Schema | Main objects | Purpose |
| --- | --- | --- |
| `raw` | `raw.FREMTPL_RAW` | Raw or lightly landed source data. In work use, this may be any source table or an existing corporate database table rather than freMTPL. |
| `mlops` | `mlops.DATASET_MANIFEST`, `mlops.DATASET_COLUMN`, `mlops.CV_SPLIT_SET`, `mlops.CV_FOLD`, `mlops.CV_SPLIT_ROW`, `mlops.MODEL_RUN`, `mlops.MODEL_RUN_DATASET`, `mlops.MODEL_RUN_SPLIT_SET`, `mlops.MODEL_RUN_METRIC`, `mlops.CV_FOLD_METRIC` | Audit layer. It answers what data was used, what split definition was used, which rows were in each test fold when materialized, and which model run consumed those assets. |
| `pricing` | `pricing.MODEL`, `pricing.RATE_PACKAGE`, `pricing.FEATURE`, `pricing.FEATURE_LEVEL_SET`, `pricing.FEATURE_LEVEL`, `pricing.TERM`, `pricing.TERM_FEATURE`, `pricing.RATE_CELL`, `pricing.RATE_CELL_LEVEL`, `pricing.MODEL_DEPLOYMENT` | Persisted pricing model and rating table structure. This is the historical record of all models, packages, terms, features, cells, and deployments. |
| `pricing_runtime` | `pricing_runtime.V_COMPILED_RATE_CELL`, `pricing_runtime.V_COMPILED_RATE_CELL_LEVEL`, `pricing_runtime.V_COMPILED_1D_RATE_BAND` | Read-optimized views for inspection, downstream rating lookups, and SQL prediction validation. |

`pricing.MODEL` is the model family. `mlops.MODEL_RUN` is one training attempt. `pricing.RATE_PACKAGE` is the published rating table output from a model run or manual adjustment. `pricing.MODEL_DEPLOYMENT` says which package is current for a slot such as UAT or PROD.

## Basic ETL Shape

The ETL should be direct: read SQL, transform a dataframe, write SQL. Keep the model-specific query and transformation in that model's folder, and keep connection mechanics in `pricing_pipeline/infra/db.py`.

In [ ]:
from __future__ import annotations

import os

import numpy as np
import pandas as pd
from sqlalchemy import text

from pricing_pipeline.infra.config import Settings
from pricing_pipeline.infra.db import get_engine


In [ ]:
SOURCE_SQL = """
SELECT
    policy_id,
    accounting_month,
    exposure,
    claim_count,
    vehicle_age,
    driver_age,
    rating_area
FROM source_schema.PolicyClaims
WHERE accounting_month >= :from_month
  AND exposure > 0
"""


def transform_source_rows(raw: pd.DataFrame) -> pd.DataFrame:
    frame = raw.copy()
    frame["log_exposure"] = np.log(frame["exposure"].astype(float).clip(lower=1e-12))
    frame["vehicle_age_band"] = pd.cut(
        frame["vehicle_age"],
        bins=[0, 1, 3, 7, 15, 100],
        right=False,
    ).astype(str)
    return frame


## Read From One SQL Server And Write To Another

`source_database` and `target_database` can be the same database or different databases on the same SQL Server. If the target is a different SQL Server instance, create a second settings object or extend the shared helper so it accepts a server override. The ETL code still only deals with engines.

In [ ]:
def run_basic_sql_etl(
    *,
    from_month: str,
    source_database: str,
    target_database: str,
    target_schema: str,
    target_table: str,
) -> int:
    settings = Settings.from_env(os.environ)

    source_engine = get_engine(settings, database=source_database)
    target_engine = get_engine(settings, database=target_database)

    raw = pd.read_sql_query(
        text(SOURCE_SQL),
        source_engine,
        params={"from_month": from_month},
    )
    transformed = transform_source_rows(raw)

    with target_engine.begin() as con:
        transformed.to_sql(
            target_table,
            con,
            schema=target_schema,
            if_exists="append",
            index=False,
            chunksize=10_000,
        )

    return len(transformed)


## Chunked Version For Larger Tables

For large tables, stream chunks from SQL Server and write each transformed chunk. This avoids holding the entire source dataset in memory.

In [ ]:
def run_chunked_sql_etl(
    *,
    from_month: str,
    source_database: str,
    target_database: str,
    target_schema: str,
    target_table: str,
    read_chunksize: int = 100_000,
) -> int:
    settings = Settings.from_env(os.environ)
    source_engine = get_engine(settings, database=source_database)
    target_engine = get_engine(settings, database=target_database)
    rows_written = 0

    with target_engine.begin() as con:
        for raw_chunk in pd.read_sql_query(
            text(SOURCE_SQL),
            source_engine,
            params={"from_month": from_month},
            chunksize=read_chunksize,
        ):
            transformed = transform_source_rows(raw_chunk)
            transformed.to_sql(
                target_table,
                con,
                schema=target_schema,
                if_exists="append",
                index=False,
                chunksize=10_000,
            )
            rows_written += len(transformed)

    return rows_written


## Where This Lives For A Real Model

For a model called `my_model`, keep the model-specific query and transform in `pricing_models/my_model/etl.py`. The DAG should call a small function from that file. Shared connection logic stays in `pricing_pipeline/infra/db.py`; shared audit writers stay under `pricing_pipeline/`.

A real DAG task can call `run_basic_sql_etl(...)` before the model training task. The model training task then reads the prepared table using its own `training_sql`.

In [ ]:
# Example DAG task body, not executed in this notebook.
def airflow_task_build_training_table() -> int:
    return run_basic_sql_etl(
        from_month="2025-01",
        source_database=os.environ["SOURCE_DATABASE"],
        target_database=os.environ["TARGET_DATABASE"],
        target_schema="mlops",
        target_table="MY_MODEL_TRAINING_DATA",
    )


## Work Authentication Note

If work SQL Server access uses Entra token auth, do not put that in every ETL task. Extend `pricing_pipeline/infra/db.py` so `get_engine(...)` can create an engine with `connect_args={"attrs_before": {1256: token_struct}}`. The constant `1256` is the Microsoft ODBC driver attribute id for an access token; it is not a secret.

Once the helper supports that mode, ETL code remains unchanged:

```python
settings = Settings.from_env(os.environ)
engine = get_engine(settings, database="ExistingWorkDatabase")
```


## Practical Rules

- Use `append` for audit/history tables unless you intentionally rebuild a scratch table.
- Use explicit SQL column lists instead of `SELECT *` for work pipelines.
- Keep source SQL and transform code close to the model that owns it.
- Keep all model runs historical; expose current state through views such as `pricing.V_CURRENT_RATE_PACKAGE`.
- Validate SQL prediction output against `SuperGLM.predict(X, offset=np.log(exposure))` with `scripts/validate_sql_prediction_against_superglm.py` before trusting a deployed rating package.